# Products Data Processing Pipeline
## Bronze → Silver → Gold → Master Dimension

### 🚀 Ready to Run!

**1. S3 Access:**
   - ✅ Unity Catalog credentials already configured!
   - Credential: `db_s3_credentials_databricks-s3-ingest-9db34`
   - No manual setup needed

**2. Data Source:**
   - S3 Bucket: `sportx-bar` (region: ap-southeast-2)
   - Path: `s3://sportx-bar/products/*.csv`
   - All CSV files in the folder will be loaded and concatenated

**3. Required columns in CSV:**
   - `product_id` - Product identifier
   - `product_name` - Full product name
   - `category` - Product category

**4. Output files:**
   All processed data will be saved to the `data/` subfolder:
   - `data/bronze_products.csv` - Raw data with metadata
   - `data/silver_products.csv` - Cleaned and transformed data
   - `data/gold_sb_dim_products.csv` - Business dimension table
   - `data/gold_dim_products.csv` - Master dimension table

**5. Run cells sequentially:**
   - Cell 4: Install s3fs
   - Cell 3: Import libraries
   - Cell 6: Define schemas
   - Cell 7: Define parameters
   - Cell 10 onwards: Run all remaining cells

---

**Import Required Libraries**

In [0]:
import pandas as pd
import numpy as np
import hashlib
import os
import re
from datetime import datetime

# Note: s3fs will be imported in the cell that needs it
print("✓ Core libraries imported")

In [0]:
# Install s3fs for S3 access
%pip install s3fs --quiet

**Load Project Utilities & Initialize Notebook Widgets**

In [0]:
# Define schema names (equivalent to utilities notebook)
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"

print(f"Schemas defined: {bronze_schema}, {silver_schema}, {gold_schema}")

In [0]:
# Define parameters (replacing dbutils.widgets)
catalog = "fmcg"
data_source = "products"

# S3 bucket with Unity Catalog credentials
s3_bucket = "sportx-bar"  # Using Unity Catalog credential: db_s3_credentials_databricks-s3-ingest-9db34

print(f"Catalog: {catalog}")
print(f"Data source: {data_source}")
print(f"S3 path: s3://{s3_bucket}/{data_source}/*.csv")

# Create data directory for outputs
os.makedirs("data", exist_ok=True)

## ✅ S3 Access Configured

**Unity Catalog Credentials Active:**
- Credential: `db_s3_credentials_databricks-s3-ingest-9db34`
- Bucket: `s3://sportx-bar/`
- Region: `ap-southeast-2` (Sydney)
- Access: Read/Write enabled

**No manual AWS credentials needed!** Databricks automatically uses Unity Catalog credentials to access S3.

---

### Ready to Run:
1. ✅ S3 credentials configured
2. ✅ Bucket: `sportx-bar`
3. ✅ Region: `ap-southeast-2`
4. ✅ Read path: `s3://sportx-bar/products/*.csv`

You can now run Cell 10 to read from S3!

## Bronze

In [0]:
# Read CSV from S3 using Spark (automatically uses Unity Catalog credentials)
# Then convert to pandas for processing

# Build S3 path
s3_path = f's3://{s3_bucket}/{data_source}/*.csv'
print(f"Reading from S3: {s3_path}")
print(f"Using Unity Catalog credential: db_s3_credentials_databricks-s3-ingest-9db34")

try:
    # Read CSV from S3 using Spark (UC credentials work automatically)
    df_spark = spark.read.csv(s3_path, header=True, inferSchema=True)
    
    # Convert to pandas DataFrame
    df = df_spark.toPandas()
    
    print(f"\n✓ Successfully loaded {len(df)} rows")
    
    # Add Bronze metadata columns
    df["read_timestamp"] = pd.Timestamp.now()
    df["file_name"] = f"{data_source}.csv"
    df["file_size"] = None
    
    print(f"\nColumns: {list(df.columns)}")
    print(f"Shape: {df.shape}")
    
except Exception as e:
    print(f"\n✗ Error reading from S3: {e}")
    print(f"\nTroubleshooting:")
    print(f"  1. Verify file exists at: {s3_path}")
    print(f"  2. Check Unity Catalog external location is configured for: s3://{s3_bucket}/")
    print(f"  3. Ensure credential has permissions: s3:GetObject, s3:ListBucket")
    raise

In [0]:
# Check data types
print("\nData types:")
print(df.dtypes)
print(f"\nShape: {df.shape}")

In [0]:
# Display first 10 rows
display(df.head(10))

In [0]:
# Save Bronze layer to CSV
bronze_path = f"data/{bronze_schema}_{data_source}.csv"
df.to_csv(bronze_path, index=False)

print(f"✓ Bronze layer saved to: {bronze_path}")
print(f"  Rows: {len(df)}")
print(f"  Columns: {len(df.columns)}")

## Silver


In [0]:
# Load Bronze data from CSV
df_bronze = pd.read_csv(bronze_path)
print(f"Loaded Bronze data: {len(df_bronze)} rows")
print(f"\nFirst 10 rows:")
display(df_bronze.head(10))

**Transformations**

- 1: Drop Duplicates

In [0]:
# Drop duplicates based on product_id
print(f'Rows before duplicates dropped: {len(df_bronze)}')
df_silver = df_bronze.drop_duplicates(subset=['product_id']).copy()
print(f'Rows after duplicates dropped: {len(df_silver)}')

- 2: Title case fix

(energy bars ---> Energy Bars, protien bars ---> Protien Bars etc)

In [0]:
# Show distinct categories before title case fix
print("Distinct categories (before):")
print(df_silver['category'].unique())

In [0]:
# Title case fix for category column
df_silver['category'] = df_silver['category'].str.title()
print("✓ Applied title case to category column")

In [0]:
# Show distinct categories after title case fix
print("\nDistinct categories (after):")
print(df_silver['category'].unique())

- 3: Fix Spelling Mistake for `Protien`

In [0]:
# Replace 'Protien' → 'Protein' in both product_name and category (case-insensitive)
df_silver['product_name'] = df_silver['product_name'].str.replace('Protien', 'Protein', case=False, regex=False)
df_silver['category'] = df_silver['category'].str.replace('Protien', 'Protein', case=False, regex=False)

print("✓ Fixed spelling: Protien → Protein")


In [0]:
# Display sample after transformations
display(df_silver.head(5))

### Standardizing Customer Attributes to Match Parent Company Data Model

In [0]:
### 1: Add division column based on category
conditions = [
    df_silver['category'] == 'Energy Bars',
    df_silver['category'] == 'Protein Bars',
    df_silver['category'] == 'Granola & Cereals',
    df_silver['category'] == 'Recovery Dairy',
    df_silver['category'] == 'Healthy Snacks',
    df_silver['category'] == 'Electrolyte Mix',
]
choices = [
    'Nutrition Bars',
    'Nutrition Bars',
    'Breakfast Foods',
    'Dairy & Recovery',
    'Healthy Snacks',
    'Hydration & Electrolytes',
]
df_silver['division'] = np.select(conditions, choices, default='Other')
print("✓ Added division column")

### 2: Extract variant from product_name (text inside parentheses)
df_silver['variant'] = df_silver['product_name'].str.extract(r'\((.*?)\)', expand=False)
print("✓ Extracted variant column")

### 3: Generate product_code and clean product_id
# Generate deterministic product_code using SHA-256 hash
df_silver['product_code'] = df_silver['product_name'].apply(
    lambda x: hashlib.sha256(str(x).encode()).hexdigest()
)

# Clean product_id: keep only numeric IDs, else set to 999999
is_numeric = df_silver['product_id'].astype(str).str.match(r'^[0-9]+$')
df_silver['product_id'] = np.where(is_numeric, df_silver['product_id'], '999999')

# Rename product_name → product
df_silver = df_silver.rename(columns={'product_name': 'product'})

print("✓ Generated product_code and cleaned product_id")
print("✓ Renamed product_name to product")

In [0]:
# Select final columns for Silver layer
silver_cols = ["product_code", "division", "category", "product", "variant", "product_id", "read_timestamp", "file_name", "file_size"]
df_silver = df_silver[silver_cols]

print("\nSelected final Silver columns:")
print(f"Columns: {list(df_silver.columns)}")

In [0]:
# Display final Silver data
print(f"\nFinal Silver data shape: {df_silver.shape}")
print(f"Columns: {list(df_silver.columns)}")
display(df_silver.head(10))

In [0]:
# Save Silver layer to CSV
silver_path = f"data/{silver_schema}_{data_source}.csv"
df_silver.to_csv(silver_path, index=False)

print(f"✓ Silver layer saved to: {silver_path}")
print(f"  Rows: {len(df_silver)}")
print(f"  Columns: {len(df_silver.columns)}")

## Gold

In [0]:
# Load Silver data
df_silver_reload = pd.read_csv(silver_path)
print(f"Loaded Silver data: {len(df_silver_reload)} rows")

# Create Gold layer: select only business columns (drop technical metadata)
gold_cols = ["product_code", "product_id", "division", "category", "product", "variant"]
df_gold = df_silver_reload[gold_cols].copy()

print(f"\nGold layer preview:")
display(df_gold.head(5))

In [0]:
# Save Gold layer to CSV (child dimension table)
gold_path = f"data/{gold_schema}_sb_dim_{data_source}.csv"
df_gold.to_csv(gold_path, index=False)

print(f"✓ Gold layer saved to: {gold_path}")
print(f"  Rows: {len(df_gold)}")
print(f"  Columns: {len(df_gold.columns)}")

## Merging Data source with parent

In [0]:
# Load child dimension (Gold layer)
df_child_products = pd.read_csv(gold_path)
merge_cols = ["product_code", "division", "category", "product", "variant"]
df_child_products = df_child_products[merge_cols].copy()

print(f"Child products loaded: {len(df_child_products)} rows")
print("\nChild products preview:")
display(df_child_products.head(5))

# Load or initialize master dimension table
master_dim_path = "data/gold_dim_products.csv"
if os.path.exists(master_dim_path):
    df_master = pd.read_csv(master_dim_path)
    print(f"\nExisting master table loaded: {len(df_master)} rows")
else:
    df_master = pd.DataFrame(columns=merge_cols)
    print("\nNo existing master table found. Creating new one.")

In [0]:
# Perform merge (upsert): update existing records, insert new ones
df_merged = df_child_products.set_index('product_code').combine_first(
    df_master.set_index('product_code')
).reset_index()

# Ensure correct column order
df_merged = df_merged[merge_cols]

# Save updated master table
df_merged.to_csv(master_dim_path, index=False)

print(f"✓ Master dimension table updated: {master_dim_path}")
print(f"  Total rows: {len(df_merged)}")
print(f"  Child rows processed: {len(df_child_products)}")
print(f"  Master rows before: {len(df_master)}")
print(f"  Master rows after: {len(df_merged)}")

print("\n" + "="*60)
print("PROCESSING COMPLETE")
print("="*60)
print(f"Bronze: data/{bronze_schema}_{data_source}.csv")
print(f"Silver: data/{silver_schema}_{data_source}.csv")
print(f"Gold:   data/{gold_schema}_sb_dim_{data_source}.csv")
print(f"Master: {master_dim_path}")